# TV3 — Offline Evaluation Runner

Notebook CPU-only để chạy evaluator trên JSONL annotation đã chuẩn bị. Notebook không gọi LLM, image backend, GPU hoặc tự tạo human labels. Bật approved Event Inventory gate để phát hiện case/event bị thiếu hoặc thừa.

Trước khi chạy, đặt file annotation đã adjudicate tại đường dẫn cấu hình bên dưới. Không dùng Test split để tuning.

In [ ]:
from pathlib import Path
from src.evaluation.evaluate import (
    evaluate_records,
    load_event_inventory_csv,
    load_jsonl,
    write_csv_report,
    write_json_report,
)

ROOT = Path.cwd()
ANNOTATION_PATH = ROOT / "outputs" / "dev_event_annotations.jsonl"  # cập nhật nếu file ở nơi khác
INVENTORY_PATH = ROOT / "data" / "dev_key_event_alignment_team_approved.csv"
OUTPUT_DIR = ROOT / "outputs" / "evaluation"
SPLIT = "Dev"  # không đổi thành Test nếu chưa có phê duyệt riêng


In [ ]:
inventory = load_event_inventory_csv(INVENTORY_PATH, split=SPLIT)
print(f"Approved cases: {len(inventory)}")
print(f"Approved events: {sum(len(ids) for ids in inventory.values())}")
print("Event counts by case:")
for case_id, event_ids in sorted(inventory.items()):
    print(f"  {case_id}: {', '.join(event_ids)}")


In [ ]:
if not ANNOTATION_PATH.is_file():
    print(f"Chưa có annotation input: {ANNOTATION_PATH}")
    print("Hãy đặt JSONL đã adjudicate đúng schema vào đường dẫn trên rồi chạy lại cell này.")
else:
    records = load_jsonl(ANNOTATION_PATH)
    report = evaluate_records(records, event_inventory=inventory)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    write_json_report(report, OUTPUT_DIR / "evaluation.json")
    write_csv_report(report, OUTPUT_DIR / "per_case.csv")
    print(f"Planned cases: {report['planned_cases']}")
    print(f"Completed outputs: {report['completed_outputs']}")
    print(f"Event Coverage: {report['metrics']['event_coverage']}")
    print(f"Event Order: {report['metrics']['event_order']}")
    print(f"Inventory status: {report['event_inventory_status']}")


## Diễn giải

- `Four Scene Rate` và `Schema Valid Rate` dùng toàn bộ planned cases làm mẫu số.
- `Event Coverage` tính trên approved events; `uncertain` vẫn nằm trong mẫu số nhưng không nằm trong tử số.
- `Event Order Accuracy` phải đọc cùng `Event Order Pair Coverage`.
- `null`/N/A có nghĩa là không có denominator hoặc thiếu dữ liệu cần thiết, không phải 0%.
- Inventory gate chỉ xác minh completeness và trạng thái phê duyệt trong CSV; không tự chứng minh semantic correctness.
- Không dùng cùng một bộ human labels cho B0 và B1 nếu hai variant có storyboard khác nhau.